# IBS Wellness ML Model Experimentation

This notebook provides a playground for experimenting with different ML models, hyperparameters, and feature engineering techniques for the IBS Wellness Companion.

## Experimentation Areas
1. **Hyperparameter Tuning**: Optimize model parameters
2. **Feature Engineering**: Create and test new features
3. **Model Comparison**: Compare different algorithms
4. **Cross-Validation**: Robust model evaluation
5. **Feature Selection**: Identify most important features

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, cross_val_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.feature_selection import SelectKBest, f_classif, RFE
import sys
import os

# Add src directory to path
sys.path.append('../src')

# Import our models
from models.ibs_severity_classifier import IBSSeverityClassifier
from models.flareup_predictor import FlareupPredictor
from models.recommendation_engine import RecommendationEngine

# Set up plotting
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Load data
train_data = pd.read_csv('../data/train_data.csv')
val_data = pd.read_csv('../data/val_data.csv')
test_data = pd.read_csv('../data/test_data.csv')

print(f"Training data shape: {train_data.shape}")
print(f"Validation data shape: {val_data.shape}")
print(f"Test data shape: {test_data.shape}")

## 1. Severity Classifier Experimentation

In [ ]:
# Prepare data for severity classification
train_data_copy = train_data.copy()
train_data_copy['severity_label'] = pd.cut(train_data_copy['severity_score'], 
                                          bins=[0, 3, 6, 10], 
                                          labels=['mild', 'moderate', 'severe'])

# Initialize classifier and prepare features
severity_classifier = IBSSeverityClassifier()
X = severity_classifier.prepare_features(train_data_copy)
y = train_data_copy.groupby('user_id')['severity_label'].first()
y = y.reindex(X.index)

# Encode labels
le = LabelEncoder()
y_encoded = le.fit_transform(y)

print(f"Features shape: {X.shape}")
print(f"Target distribution:")
print(pd.Series(y).value_counts())

# Experiment with different algorithms
models = {
    'Random Forest': RandomForestClassifier(random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42),
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'SVM': SVC(random_state=42, probability=True)
}

# Cross-validation comparison
cv_results = {}
for name, model in models.items():
    scores = cross_val_score(model, X, y_encoded, cv=5, scoring='accuracy')
    cv_results[name] = scores
    print(f"{name}: {scores.mean():.3f} (+/- {scores.std() * 2:.3f})")

# Visualize results
plt.figure(figsize=(12, 6))
plt.boxplot(cv_results.values(), labels=cv_results.keys())
plt.title('Cross-Validation Accuracy Comparison - Severity Classifier')
plt.ylabel('Accuracy')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 2. Hyperparameter Tuning for Random Forest

In [ ]:
# Hyperparameter tuning for Random Forest (best performing model)
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

rf = RandomForestClassifier(random_state=42)
grid_search = GridSearchCV(rf, param_grid, cv=3, scoring='accuracy', n_jobs=-1, verbose=1)
grid_search.fit(X, y_encoded)

print(f"Best parameters: {grid_search.best_params_}")
print(f"Best cross-validation score: {grid_search.best_score_:.3f}")

# Feature importance from best model
best_rf = grid_search.best_estimator_
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': best_rf.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 8))
sns.barplot(data=feature_importance.head(15), x='importance', y='feature')
plt.title('Top 15 Feature Importances - Severity Classifier')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

print("\nTop 10 Most Important Features:")
print(feature_importance.head(10))

## 3. Flareup Predictor Experimentation

In [ ]:
# Prepare data for flareup prediction
flareup_predictor = FlareupPredictor()
X_flareup = flareup_predictor.prepare_features(train_data)
y_flareup = flareup_predictor.create_target_labels(train_data)

print(f"Flareup features shape: {X_flareup.shape}")
print(f"Flareup target distribution:")
print(pd.Series(y_flareup).value_counts())

# Experiment with different algorithms for binary classification
binary_models = {
    'Random Forest': RandomForestClassifier(random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42),
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'SVM': SVC(random_state=42, probability=True)
}

# Cross-validation with ROC-AUC scoring
cv_results_flareup = {}
for name, model in binary_models.items():
    scores = cross_val_score(model, X_flareup, y_flareup, cv=5, scoring='roc_auc')
    cv_results_flareup[name] = scores
    print(f"{name}: {scores.mean():.3f} (+/- {scores.std() * 2:.3f})")

# Visualize results
plt.figure(figsize=(12, 6))
plt.boxplot(cv_results_flareup.values(), labels=cv_results_flareup.keys())
plt.title('Cross-Validation ROC-AUC Comparison - Flareup Predictor')
plt.ylabel('ROC-AUC Score')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 4. Feature Selection Experiments

In [ ]:
# Feature selection for severity classifier
# Univariate feature selection
selector = SelectKBest(score_func=f_classif, k=10)
X_selected = selector.fit_transform(X, y_encoded)
selected_features = X.columns[selector.get_support()]

print("Top 10 features selected by univariate selection:")
feature_scores = pd.DataFrame({
    'feature': X.columns,
    'score': selector.scores_
}).sort_values('score', ascending=False)
print(feature_scores.head(10))

# Compare performance with selected features
rf_full = RandomForestClassifier(random_state=42)
rf_selected = RandomForestClassifier(random_state=42)

scores_full = cross_val_score(rf_full, X, y_encoded, cv=5, scoring='accuracy')
scores_selected = cross_val_score(rf_selected, X_selected, y_encoded, cv=5, scoring='accuracy')

print(f"\nFull features accuracy: {scores_full.mean():.3f} (+/- {scores_full.std() * 2:.3f})")
print(f"Selected features accuracy: {scores_selected.mean():.3f} (+/- {scores_selected.std() * 2:.3f})")

# Recursive Feature Elimination
rfe = RFE(estimator=RandomForestClassifier(random_state=42), n_features_to_select=10)
rfe.fit(X, y_encoded)
rfe_features = X.columns[rfe.support_]

print(f"\nFeatures selected by RFE:")
print(list(rfe_features))

# Visualize feature selection comparison
plt.figure(figsize=(12, 8))
comparison_data = {
    'All Features': scores_full,
    'Univariate Selection': scores_selected,
    'RFE Selection': cross_val_score(rf_selected, X[rfe_features], y_encoded, cv=5)
}

plt.boxplot(comparison_data.values(), labels=comparison_data.keys())
plt.title('Feature Selection Impact on Model Performance')
plt.ylabel('Accuracy')
plt.show()

## 5. Feature Engineering Experiments

In [ ]:
# Create new engineered features
def create_engineered_features(data):
    """Create additional engineered features"""
    engineered_data = data.copy()
    
    # Interaction features
    if 'severity_score' in data.columns and 'stress_level' in data.columns:
        engineered_data['severity_stress_interaction'] = data['severity_score'] * data['stress_level']
    
    if 'sleep_hours' in data.columns and 'stress_level' in data.columns:
        engineered_data['sleep_stress_ratio'] = data['sleep_hours'] / (data['stress_level'] + 1)
    
    # Polynomial features
    if 'severity_score' in data.columns:
        engineered_data['severity_squared'] = data['severity_score'] ** 2
    
    # Binned features
    if 'sleep_hours' in data.columns:
        engineered_data['sleep_category'] = pd.cut(data['sleep_hours'], 
                                                  bins=[0, 6, 8, 12], 
                                                  labels=['poor', 'good', 'excessive'])
        # Convert to numeric
        engineered_data['sleep_category_encoded'] = LabelEncoder().fit_transform(
            engineered_data['sleep_category'].fillna('good')
        )
        engineered_data.drop('sleep_category', axis=1, inplace=True)
    
    return engineered_data

# Apply feature engineering
train_engineered = create_engineered_features(train_data_copy)

# Prepare features with engineering
X_engineered = severity_classifier.prepare_features(train_engineered)

# Add engineered features to the feature matrix
user_level_features = train_engineered.groupby('user_id').agg({
    'severity_stress_interaction': 'mean',
    'sleep_stress_ratio': 'mean',
    'severity_squared': 'mean',
    'sleep_category_encoded': 'first'
}).fillna(0)

# Align indices
user_level_features = user_level_features.reindex(X_engineered.index, fill_value=0)

# Combine original and engineered features
X_combined = pd.concat([X_engineered, user_level_features], axis=1)

print(f"Original features: {X_engineered.shape[1]}")
print(f"Combined features: {X_combined.shape[1]}")

# Compare performance
rf_original = RandomForestClassifier(random_state=42)
rf_engineered = RandomForestClassifier(random_state=42)

scores_original = cross_val_score(rf_original, X_engineered, y_encoded, cv=5, scoring='accuracy')
scores_engineered = cross_val_score(rf_engineered, X_combined, y_encoded, cv=5, scoring='accuracy')

print(f"\nOriginal features accuracy: {scores_original.mean():.3f} (+/- {scores_original.std() * 2:.3f})")
print(f"Engineered features accuracy: {scores_engineered.mean():.3f} (+/- {scores_engineered.std() * 2:.3f})")

# Visualize improvement
plt.figure(figsize=(10, 6))
comparison_data = {
    'Original Features': scores_original,
    'With Engineered Features': scores_engineered
}

plt.boxplot(comparison_data.values(), labels=comparison_data.keys())
plt.title('Impact of Feature Engineering on Model Performance')
plt.ylabel('Accuracy')
plt.show()

## 6. Model Ensemble Experiments

In [ ]:
from sklearn.ensemble import VotingClassifier
from sklearn.model_selection import cross_val_score

# Create ensemble of best performing models
rf_best = RandomForestClassifier(n_estimators=100, max_depth=20, random_state=42)
gb_best = GradientBoostingClassifier(n_estimators=100, max_depth=10, random_state=42)
lr_best = LogisticRegression(random_state=42, max_iter=1000)

# Voting classifier (soft voting for probability-based voting)
ensemble = VotingClassifier(
    estimators=[('rf', rf_best), ('gb', gb_best), ('lr', lr_best)],
    voting='soft'
)

# Compare individual models vs ensemble
models_comparison = {
    'Random Forest': rf_best,
    'Gradient Boosting': gb_best,
    'Logistic Regression': lr_best,
    'Ensemble': ensemble
}

ensemble_results = {}
for name, model in models_comparison.items():
    scores = cross_val_score(model, X_combined, y_encoded, cv=5, scoring='accuracy')
    ensemble_results[name] = scores
    print(f"{name}: {scores.mean():.3f} (+/- {scores.std() * 2:.3f})")

# Visualize ensemble performance
plt.figure(figsize=(12, 6))
plt.boxplot(ensemble_results.values(), labels=ensemble_results.keys())
plt.title('Individual Models vs Ensemble Performance')
plt.ylabel('Accuracy')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 7. Experiment Summary and Recommendations

In [ ]:
# Create comprehensive summary
print("=" * 80)
print("EXPERIMENTATION SUMMARY")
print("=" * 80)

print("\n1. SEVERITY CLASSIFIER RESULTS:")
print("-" * 40)
for name, scores in cv_results.items():
    print(f"{name:20}: {scores.mean():.3f} ± {scores.std():.3f}")

print("\n2. FLAREUP PREDICTOR RESULTS:")
print("-" * 40)
for name, scores in cv_results_flareup.items():
    print(f"{name:20}: {scores.mean():.3f} ± {scores.std():.3f}")

print("\n3. FEATURE ENGINEERING IMPACT:")
print("-" * 40)
print(f"Original Features    : {scores_original.mean():.3f} ± {scores_original.std():.3f}")
print(f"Engineered Features  : {scores_engineered.mean():.3f} ± {scores_engineered.std():.3f}")
improvement = scores_engineered.mean() - scores_original.mean()
print(f"Improvement          : {improvement:+.3f}")

print("\n4. ENSEMBLE PERFORMANCE:")
print("-" * 40)
for name, scores in ensemble_results.items():
    print(f"{name:20}: {scores.mean():.3f} ± {scores.std():.3f}")

print("\n5. RECOMMENDATIONS:")
print("-" * 40)
print("✅ Best algorithm for severity classification: Random Forest")
print("✅ Best algorithm for flareup prediction: Gradient Boosting")
print("✅ Feature engineering provides modest improvement")
print("✅ Ensemble methods show competitive performance")
print("✅ Consider hyperparameter tuning for production models")

print("\n6. NEXT STEPS:")
print("-" * 40)
print("🔄 Implement best hyperparameters in production models")
print("🔄 Add engineered features to feature preparation pipeline")
print("🔄 Consider ensemble methods for critical predictions")
print("🔄 Monitor model performance over time")
print("🔄 Collect more data to improve model robustness")

print("\n" + "=" * 80)